In [1]:
import sys
sys.path.append('../../')

import pandas as pd
from src.io.save_artifacts import load_parquet, save_parquet
from src.cleaning.parse_percentages import parse_percentages
from src.cleaning.parse_dates import parse_dates
from src.cleaning.normalize_strings import normalize_strings

df = load_parquet('../../data/processed/lending_club_no_leakage.parquet')
print(f"Başlangıç shape: {df.shape}")
print(f"Kolonlar: {df.columns.tolist()}")

📂 Parquet okunuyor: ../../data/processed/lending_club_no_leakage.parquet
✅ Okundu!
   Satır : 1,345,350
   Sütun : 114
   Süre  : 0.3 saniye
Başlangıç shape: (1345350, 114)
Kolonlar: ['row_id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'annual_inc_joint', 'dti_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total

In [2]:
# Yüzde kolonları
for k in ['int_rate', 'revol_util']:
    if k in df.columns:
        print(f"{k}: dtype={df[k].dtype}, örnek={df[k].dropna().iloc[0]}")

int_rate: dtype=float32, örnek=13.989999771118164
revol_util: dtype=float32, örnek=29.700000762939453


In [3]:
# Yüzde dönüşümü 
yuzde_kolonlar = []
for k in ['int_rate', 'revol_util']:
    if k in df.columns and df[k].dtype == object:
        yuzde_kolonlar.append(k)

if yuzde_kolonlar:
    df = parse_percentages(df, kolonlar=yuzde_kolonlar, bolme=False)
    print(f"✅ Dönüştürüldü: {yuzde_kolonlar}")
else:
    print("ℹ️  Yüzde dönüşümü gerekmiyor — kolonlar zaten sayısal")

ℹ️  Yüzde dönüşümü gerekmiyor — kolonlar zaten sayısal


In [4]:
# term kolonundaki boşluğu temizle
if 'term' in df.columns:
    df['term'] = df['term'].astype(str).str.strip()
    print("✅ term strip() uygulandı")
    print(df['term'].value_counts().to_string())

✅ term strip() uygulandı
term
36 months    1020768
60 months     324582


In [5]:
# Tarih kolonlarını dönüştür
# referans_tarih: veri setindeki son tarih — 2018 sonu
tarih_kolonlar = [c for c in ['issue_d', 'earliest_cr_line']
                  if c in df.columns]

if tarih_kolonlar:
    df = parse_dates(
        df,
        kolonlar=tarih_kolonlar,
        referans_tarih='2018-12-01',
        format='%b-%Y'
    )
    print(f"\nShape: {df.shape}")
else:
    print("ℹ️  Tarih kolonu bulunamadı")

 Tarih Dönüşümü 
   ✅ issue_d → issue_d_YIL, issue_d_AY, issue_d_AY_FARK
   ✅ earliest_cr_line → earliest_cr_line_YIL, earliest_cr_line_AY, earliest_cr_line_AY_FARK

Shape: (1345350, 118)


In [6]:
df = normalize_strings(df, lower=False, strip=True)
print(f"\nShape: {df.shape}")

✅ String normalize: 0 kolon işlendi

Shape: (1345350, 118)


In [8]:
# Format sonrası kontrol
print("── Format Kontrol ───────────────────────────")
kontrol_kolonlar = ['int_rate', 'revol_util', 'term',
                    'issue_d_YIL', 'issue_d_AY',
                    'issue_d_AY_FARK',
                    'earliest_cr_line_YIL',
                    'earliest_cr_line_AY_FARK']

for k in kontrol_kolonlar:
    if k in df.columns:
        ornek = df[k].dropna().iloc[0]
        # Sayısal ise .2f, değilse olduğu gibi göster
        if pd.api.types.is_numeric_dtype(df[k]):
            ornek_str = f"{ornek:.2f}"
        else:
            ornek_str = str(ornek)
        print(f"   {k:<30} dtype={str(df[k].dtype):<10}  örnek={ornek_str}")

── Format Kontrol ───────────────────────────
   int_rate                       dtype=float32     örnek=13.99
   revol_util                     dtype=float32     örnek=29.70
   term                           dtype=str         örnek=36 months
   issue_d_YIL                    dtype=float32     örnek=2015.00
   issue_d_AY                     dtype=float32     örnek=12.00
   issue_d_AY_FARK                dtype=float32     örnek=36.53
   earliest_cr_line_YIL           dtype=float32     örnek=2003.00
   earliest_cr_line_AY_FARK       dtype=float32     örnek=186.70


In [9]:
save_parquet(df, '../../data/processed/lending_club_formatted.parquet')

print("=" * 50)
print("  02_format_donusturme TAMAMLANDI")
print("=" * 50)
print(f"  Satır : {df.shape[0]:,}")
print(f"  Sütun : {df.shape[1]}")
print("→ Sıradaki: 03_eksik_deger_doldurma.ipynb")

💾 Parquet kaydediliyor: ../../data/processed/lending_club_formatted.parquet
✅ Kaydedildi!
   Satır  : 1,345,350
   Sütun  : 118
   Boyut  : 104.3 MB
   Süre   : 3.9 saniye
  02_format_donusturme TAMAMLANDI
  Satır : 1,345,350
  Sütun : 118
→ Sıradaki: 03_eksik_deger_doldurma.ipynb
